In [ ]:
import pandas as pd
import numpy as np
import math
from scipy import stats as scipy_stats
from typing import Union, List, Tuple, Callable

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)

    Parameters
    ----------
    df         : input DataFrame
    target_col : name of the column to discretise

    Returns
    -------
    dict  {bin_id (int, 0-based): [row_index, ...]}
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    # Cut into n_bins equal-width intervals; labels are 0-based integer bin IDs
    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
# Sample DataFrame
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(100),
    "feature_b": np.random.rand(100),
    "target":    np.random.normal(loc=50, scale=15, size=100)
})

bin_map, n_bins = bin_target(df, target_col="target")

print(f"Scott's rule → {n_bins} bins\n")
for bin_id, indices in bin_map.items():
    print(f"  Bin {bin_id:>2d} ({len(indices):>3d} items): {indices[:5]}{'...' if len(indices) > 5 else ''}")

In [ ]:
DIST_MAP = {
    "normal":      scipy_stats.norm,
    "gaussian":    scipy_stats.norm,
    "exponential": scipy_stats.expon,
    "beta":        scipy_stats.beta,
    "gamma":       scipy_stats.gamma,
    "uniform":     scipy_stats.uniform,
    "lognormal":   scipy_stats.lognorm,
}

def make_pdf(
    spec: Union[
        Tuple[str, dict],
        List[Tuple[str, dict, float]]
    ]
) -> Callable:
    """
    Single distribution:
        spec = ("normal", {"loc": 0, "scale": 1})

    Mixture (weights are auto-normalised):
        spec = [
            ("normal",      {"loc": 0, "scale": 1},  0.6),
            ("exponential", {"scale": 2},             0.4),
        ]
    """
    if isinstance(spec, tuple):
        name, params = spec
        dist = DIST_MAP[name.lower()]
        return lambda x: dist.pdf(x, **params)

    total_w    = sum(w for _, _, w in spec)
    components = [(DIST_MAP[n.lower()], p, w / total_w) for n, p, w in spec]

    def mixture_pdf(x):
        return sum(w * d.pdf(x, **p) for d, p, w in components)

    return mixture_pdf

In [ ]:
def least_cardinality_bin(bin_map: dict) -> Tuple[int, int]:
    """Return (bin_id, cardinality) of the bin with the fewest items."""
    bin_id = min(bin_map, key=lambda b: len(bin_map[b]))
    return bin_id, len(bin_map[bin_id])


def _bin_centers(df: pd.DataFrame, target_col: str, n_bins: int) -> np.ndarray:
    """Midpoint of each equal-width bin over the target column range."""
    lo, hi = df[target_col].min(), df[target_col].max()
    edges  = np.linspace(lo, hi, n_bins + 1)
    return (edges[:-1] + edges[1:]) / 2

In [ ]:
def max_peak_samples(
    bin_map:       dict,
    df:            pd.DataFrame,
    target_col:    str,
    n_bins:        int,
    pdf:           Callable,
    min_total_pct: float = 0.0,
) -> Tuple[int, List[int], bool]:
    """
    Binary search over [1, n] (n = min-cardinality bin) for the largest k such
    that sampling k items from every peak bin leaves all other bins able to
    supply their proportional quota.

    min_total_pct : minimum required total samples as a fraction of len(df),
                    e.g. 0.4 = at least 40% of all rows must be covered.

    Returns
    -------
    k_max     : largest feasible k (0 if infeasible)
    peak_bins : bin_ids at the distribution peak (multimodal -> multiple ids)
    feasible  : True iff a valid k satisfying both constraints exists
    """
    centers   = _bin_centers(df, target_col, n_bins)
    bin_ids   = sorted(bin_map.keys())
    pdf_vals  = np.array([pdf(centers[i]) for i in bin_ids])
    peak_pdf  = pdf_vals.max()
    peak_bins = [bid for bid, pv in zip(bin_ids, pdf_vals)
                 if np.isclose(pv, peak_pdf)]

    _, n      = least_cardinality_bin(bin_map)
    min_total = round(min_total_pct * len(df))

    def total_samples(k: int) -> int:
        return sum(round(k * pv / peak_pdf) for pv in pdf_vals)

    def within_capacity(k: int) -> bool:
        return all(
            round(k * pv / peak_pdf) <= len(bin_map[bid])
            for bid, pv in zip(bin_ids, pdf_vals)
        )

    # k_hi: largest k that fits within all bin capacities
    lo, hi, k_hi = 1, n, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        if within_capacity(mid):
            k_hi = mid
            lo   = mid + 1
        else:
            hi   = mid - 1

    if k_hi == 0:
        return 0, peak_bins, False

    # k_lo: smallest k whose total meets min_total (monotone increasing in k)
    lo, hi, k_lo = 1, k_hi, k_hi + 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if total_samples(mid) >= min_total:
            k_lo = mid
            hi   = mid - 1
        else:
            lo   = mid + 1

    feasible = k_lo <= k_hi
    return k_hi if feasible else 0, peak_bins, feasible

In [ ]:
def sanity_check(
    bin_map:       dict,
    df:            pd.DataFrame,
    target_col:    str,
    n_bins:        int,
    pdf:           Callable,
    k:             int,
    min_total_pct: float = 0.0,
) -> pd.DataFrame:
    """
    Per-bin report of available vs required cardinality, plus a TOTAL row.
    min_total_pct is expressed as a fraction of len(df), e.g. 0.4 = 40%.
    """
    centers   = _bin_centers(df, target_col, n_bins)
    bin_ids   = sorted(bin_map.keys())
    pdf_vals  = np.array([pdf(centers[i]) for i in bin_ids])
    peak_pdf  = pdf_vals.max()
    min_total = round(min_total_pct * len(df))

    rows = []
    for bid, pv in zip(bin_ids, pdf_vals):
        required  = round(k * pv / peak_pdf)
        available = len(bin_map[bid])
        rows.append({
            "bin_id":     bid,
            "center":     round(centers[bid], 4),
            "pdf_value":  round(pv, 6),
            "available":  available,
            "required":   required,
            "sufficient": available >= required,
        })

    report    = pd.DataFrame(rows)
    total_req = report["required"].sum()
    total_avl = report["available"].sum()

    report = pd.concat([report, pd.DataFrame([{
        "bin_id":     "TOTAL",
        "center":     "",
        "pdf_value":  "",
        "available":  total_avl,
        "required":   total_req,
        "sufficient": total_req >= min_total,
    }])], ignore_index=True)

    bins_ok  = report.iloc[:-1]["sufficient"].all()
    total_ok = total_req >= min_total
    status   = "PASSED ✓" if (bins_ok and total_ok) else "FAILED ✗"
    print(f"Sanity check {status}")
    print(f"  Bin capacities : {report.iloc[:-1]['sufficient'].sum()}/{len(report)-1} sufficient")
    print(f"  Total samples  : {total_req} / {len(df)} rows  "
          f"({100*total_req/len(df):.1f}%,  min required: {min_total_pct*100:.0f}% = {min_total}) "
          f"{'✓' if total_ok else '✗'}")
    return report

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(500),
    "target":    np.random.normal(loc=50, scale=15, size=500),
})
bin_map, n_bins = bin_target(df, "target")
pdf = make_pdf(("gaussian", {"loc": 50, "scale": 15}))

lc_bin, lc_count = least_cardinality_bin(bin_map)
print(f"Least-cardinality bin: {lc_bin}  (n={lc_count})  |  Total rows: {len(df)}\n")

# case 1: achievable (40% of 500 = 200 rows)
PCT = 0.40
k_max, peak_bins, ok = max_peak_samples(bin_map, df, "target", n_bins, pdf, min_total_pct=PCT)
print(f"[min_total_pct={PCT:.0%}]  feasible={ok}  k_max={k_max}  peak_bins={peak_bins}\n")
report = sanity_check(bin_map, df, "target", n_bins, pdf, k_max, min_total_pct=PCT)
display(report)

# case 2: infeasible (99% of 500 = 495 rows — impossible from small bins)
print("\n── Infeasible min_total_pct ──")
PCT_BIG = 0.99
k_max2, peak_bins2, ok2 = max_peak_samples(bin_map, df, "target", n_bins, pdf, min_total_pct=PCT_BIG)
print(f"[min_total_pct={PCT_BIG:.0%}]  feasible={ok2}  k_max={k_max2}")

# case 3: bimodal mixture with 50% coverage requirement
print("\n── Bimodal mixture ──")
pdf_mix = make_pdf([
    ("normal", {"loc": 35, "scale": 8},  0.5),
    ("normal", {"loc": 65, "scale": 8},  0.5),
])
k_mix, peaks_mix, ok_mix = max_peak_samples(bin_map, df, "target", n_bins, pdf_mix, min_total_pct=0.50)
print(f"[min_total_pct=50%]  feasible={ok_mix}  k_max={k_mix}  peak_bins={peaks_mix}\n")
report_mix = sanity_check(bin_map, df, "target", n_bins, pdf_mix, k_mix, min_total_pct=0.50)
display(report_mix)